In [ ]:
import jax
import jax.numpy as jnp
from functools import partial
from typing import Callable, Tuple, Optional
import jax.scipy as jsp
import matplotlib.pyplot as plt
import jax.random as jr
def softabs(l, alpha):
    return jnp.where(jnp.abs(l) < 1e-6, 1.0 / alpha, l / jnp.tanh(l * alpha))

def J_matrix(eigvals, alpha):
    f = softabs(eigvals, alpha)
    li, lj = eigvals[:, None], eigvals[None, :]
    fi, fj = f[:, None], f[None, :]
    denom = li - lj
    safe_denom = jnp.where(jnp.abs(denom) < 1e-10, 1.0, denom)   # avoid 0/0 before masking
    J = (fi - fj) / safe_denom
    al = alpha * eigvals
    deriv = jnp.where(jnp.abs(al) < 1e-8, 0.0, 1.0/jnp.tanh(al) - al/jnp.sinh(al)**2)
    mask = jnp.abs(li - lj) < 1e-10
    J = jnp.where(mask, deriv[:, None], J)
    return J

def function(x):
    return x[0]**3 - x[1]**3

def H_tilde(H, alpha):
    eigvals, Q = jnp.linalg.eigh(H)
    neweigvals = softabs(eigvals, alpha)
    return Q @ jnp.diag(neweigvals) @ Q.T


def dlog_softabs_det(eigvals, Q, dH_dtheta, alpha):
    f = softabs(eigvals, alpha)
    R = 1.0 / f
    J = J_matrix(eigvals, alpha)
    weight = R * jnp.diag(J)          # only need J's diagonal: J_ii
    C = (Q * weight) @ Q.T           # scale columns of Q, then Q^T  -- O(N^3), same cost as before
    D = len(eigvals)
    #grad = jnp.array([jnp.trace(C @ dH_dtheta[:, :, k]) for k in range(D)])
    grad = jnp.einsum('ij,ijk->k', C, dH_dtheta)
    return grad

def grad_of_momentum_hamiltonian(eigvals, Q, p, dH_dtheta, alpha):
    mul = Q.T @ p
    D = jnp.diag(mul/softabs(eigvals, alpha))
    C = Q @ D @ J_matrix(eigvals, alpha) @ D @ Q.T
    grad = -jnp.einsum('ij, ijk -> k', C, dH_dtheta)
    return grad


def Hamiltonian(position, momentum, negative_logdensity, hess_at_position, a):
    H_smooth = H_tilde(hess_at_position, a)
    _, logdet = jnp.linalg.slogdet(H_smooth)
    return 0.5 * momentum.T @ jnp.linalg.solve(H_smooth, momentum) + 0.5*logdet + negative_logdensity(position)

def one_leapfrog_step(position, momentum, negative_logdensity, dt, a, key):
    old_position = position
    old_momentum = momentum
    hess = jax.hessian(negative_logdensity)
    H_initial = hess(position)
    old_hamiltonian = Hamiltonian(position, momentum, negative_logdensity, H_initial, a)
    initial_eigvals, initial_Q = jnp.linalg.eigh(H_initial)
    initial_dH_dtheta = jax.jacfwd(hess)(position)
    grad_phi_initial = 0.5*dlog_softabs_det(initial_eigvals, initial_Q, initial_dH_dtheta, a) + jax.grad(negative_logdensity)(position)
    momentum = momentum - dt*0.5*grad_phi_initial

    rho = momentum
    error = 10
    tolerance = 0.1
    max_iterations = 100
    iteration_count = 0
    while(error > tolerance):
        partial_tau_partial_q = 0.5*grad_of_momentum_hamiltonian(initial_eigvals, initial_Q, momentum, initial_dH_dtheta, a)
        new_momentum = rho - dt*0.5*partial_tau_partial_q
        error = jnp.max(jnp.abs(momentum - new_momentum))
        momentum = new_momentum
        iteration_count = iteration_count + 1
        if(iteration_count > max_iterations):
            return old_position, old_momentum, True, 1
    
    sigma = position
    error = 10
    iteration_count = 0
    H_tilde_old = H_tilde(H_initial, a)
    partial_tau_partial_p_old = jnp.linalg.solve(H_tilde_old, momentum)
    while(error > tolerance):
        H_new = hess(position)
        H_tilde_new = H_tilde(H_new, a)
        partial_tau_partial_p_new = jnp.linalg.solve(H_tilde_new, momentum)
        new_position = sigma + dt*0.5*(partial_tau_partial_p_old + partial_tau_partial_p_new)
        error = jnp.max(jnp.abs(position - new_position))
        position = new_position
        iteration_count = iteration_count + 1
        if(iteration_count > max_iterations):
            return old_position, old_momentum, True, 1

    H_final = hess(position)
    final_eigvals, final_Q = jnp.linalg.eigh(H_final)
    final_dH_dtheta = jax.jacfwd(hess)(position)
    partial_tau_partial_q_final = 0.5*grad_of_momentum_hamiltonian(final_eigvals, final_Q, momentum, final_dH_dtheta, a)
    momentum = momentum - dt*0.5*partial_tau_partial_q_final

    grad_phi_final= 0.5*dlog_softabs_det(final_eigvals, final_Q, final_dH_dtheta, a) + jax.grad(negative_logdensity)(position)
    momentum = momentum - dt*0.5*grad_phi_final
    new_hamiltonian = Hamiltonian(position, momentum, negative_logdensity, H_final, a)
    accept_prob = jnp.minimum(1.0, jnp.exp(old_hamiltonian - new_hamiltonian))
    accept_prob = jnp.where(jnp.isnan(accept_prob), 0.0, accept_prob)
    was_rejected = jr.uniform(key) >= accept_prob
    position = jnp.where(was_rejected, old_position, position)
    momentum = jnp.where(was_rejected, old_momentum, momentum)
    return position, momentum, was_rejected, accept_prob


def one_overall_step(position, momentum, negative_logdensity, step_size, num_steps, a, key):
    accept_prob_arr = jnp.zeros(num_steps)
    for i in range (0, num_steps):
        key, subkey = jr.split(key)
        position, momentum, _, accept_prob = one_leapfrog_step(position, momentum, negative_logdensity, step_size, a, subkey)
        accept_prob_arr = accept_prob_arr.at[i].set(accept_prob)
    return position, momentum, accept_prob_arr







ValueError: vmap was requested to map its argument along axis 0, which implies that its rank should be at least 1, but is only 0 (its shape is ())